# Calibration, sensor failures and latency (CS637)

Run top to bottom. Scripts live in `MyDrive/CS637/extensions_out/scripts/` (copies of this folder's `.py` files).

| Section | Needs | Runtime | Time |
|---|---|---|---|
| 2 Latency and model size | authors' checkpoints only | CPU is enough | ~3 min |
| 3 Sensor failures | raw PEMSD4 in `data/raw/` + `process_pems.py` from the reproduction scripts | T4 GPU | ~2 min data, ~3 min check, a few hours for the full run |
| 4 Coverage + conformal | the checkpoints (and the shared prediction files, as a cross-check) | T4 GPU | ~5 min |

Switching runtime wipes `/content`: after a switch, run section 1 again (and the data cell in section 3).
All outputs go to `MyDrive/CS637/extensions_out/`.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
ROOT = '/content/drive/MyDrive/CS637'
OUT = f'{ROOT}/extensions_out'
SCRIPTS = f'{OUT}/scripts'
REPRO_SCRIPTS = f'{ROOT}/reproduction/scripts'   # process_pems.py
for d in (OUT, SCRIPTS, f'{OUT}/latency', f'{OUT}/sensor_failure', f'{OUT}/coverage'):
    os.makedirs(d, exist_ok=True)
print(sorted(os.listdir(SCRIPTS)))   # expect coverage_conformal.py, latency.py, checkpoint_utils.py, sensor_failure.py

In [ ]:
# Our own copy of the three PEMSD4 Quantile variants (about 25 MB, with checkpoints).
# Cloned to /content so we never touch the shared uq4phy_traffic_network folder.
VARIANTS = ['quantile_model_PEMSD4', 'gpde_quantile_model_PEMSD4', 'gpde_quantile_phy_model_PEMSD4']
if not os.path.exists('/content/authors'):
    !git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/tianshubao1/uq4phy_traffic_network.git /content/authors
    !cd /content/authors && git sparse-checkout init --cone && git sparse-checkout set {' '.join(VARIANTS)} && git checkout -q
VDIRS = [f'/content/authors/{v}' for v in VARIANTS]
for v in VDIRS:
    print(v, sorted(os.listdir(f'{v}/models')))
args = ' '.join(f'--variant {v}' for v in VDIRS)   # used by sections 2, 3 and 4

## 2. Latency and model size (CPU)

The script hides the GPU, so this measures the CPU (note the CPU model in `latency_meta.json`). Defaults: batches 1/16/64, 10 timed passes each, about 3 minutes in total. Batch 256 (`--batches 1,16,64,256`) adds about 10 minutes per model on Colab's CPU and is not needed for the real-time argument.

`peak_rss_MB` in `model_size.csv` is the whole process (Python + PyTorch + the loader's batch-256 warm-up), not the model's memory; don't report it as such.

In [ ]:
!python {SCRIPTS}/latency.py {args} --out {OUT}/latency
import pandas as pd
display(pd.read_csv(f'{OUT}/latency/model_size.csv'))
display(pd.read_csv(f'{OUT}/latency/latency.csv'))

## 3. Sensor failures

Needs processed PEMSD4 (`train/val/test.npz`, channels [speed, occupancy, flow], 11871/1676/3376 samples). The next cell builds it on local disk with `process_pems.py` (reproduction scripts) from `data/raw/pems04.npz` (writing large files to the shared Drive fails), then checks it.

In [ ]:
PEMSD4_DIR = '/content/data/PEMSD4'
!mkdir -p {PEMSD4_DIR}
!python {REPRO_SCRIPTS}/process_pems.py --raw {ROOT}/data/raw/pems04.npz --out {PEMSD4_DIR}
!ls -la {PEMSD4_DIR}

import numpy as np
t = np.load(f'{PEMSD4_DIR}/test.npz')
print(t['x'].shape, t['y'].shape)     # expect (3376, 12, 307, 3) twice
print('mean speed', t['x'][..., 0].mean(), 'mean occupancy', t['x'][..., 1].mean())  # speed ~60 mph, occupancy < 1

In [ ]:
# Sanity check first: clean test MAE should be close to the authors' log at the saved epoch
#   quantile 2.6702 | gpde_quantile 2.1338 | gpde_quantile_phy 2.0189
# (small differences are expected: their evaluation also counts padded copies of the last sample)
!python {SCRIPTS}/sensor_failure.py --dataset_dir {PEMSD4_DIR} {args} --out {OUT}/sensor_failure/clean_check --clean_only

In [ ]:
# Full experiment: dead and stuck sensors, 5/10/20/40 %, 3 random sensor sets, with and without the fix.
# Roughly 49 test-set passes per model; check the time of the first lines and reduce --seeds if needed.
!python {SCRIPTS}/sensor_failure.py --dataset_dir {PEMSD4_DIR} {args} --out {OUT}/sensor_failure

In [ ]:
from IPython.display import Image, display
for f in sorted(glob.glob(f'{OUT}/sensor_failure/*.png')):
    print(f); display(Image(f))

### 3b. Improved fix: impute dead sensors only

The first run showed that replacing *stuck* sensors by their neighbours' mean is worse than keeping their stale reading. With `--detect zero` only all-zero (dead) windows are replaced. Expected: dead results equal the first run's fix rows, and stuck results equal its no-fix rows. This run confirms that on seed 0 (fix only, about a sixth of the full run); if it matches, the 3-seed numbers for the improved fix come straight from the first run's CSV.

In [ ]:
import shutil
!python {SCRIPTS}/sensor_failure.py --dataset_dir {PEMSD4_DIR} {args} --out /content/sensor_failure_dead_only \
    --detect zero --fix_only --seeds 1
for f in glob.glob('/content/sensor_failure_dead_only/*'):
    shutil.copy(f, f'{OUT}/sensor_failure/dead_only_' + os.path.basename(f))
print(sorted(os.listdir(f'{OUT}/sensor_failure')))

## 4. Coverage and conformal calibration

Two sources of predictions, same format (keys `val_pred`, `test_pred`, `val_truth`, `test_truth`):
- our own, made straight from the checkpoints with `make_predictions.py` (needs `PEMSD4_DIR` from section 3; about a minute per model on GPU);
- the shared files in `outputs/pemsd4/`, if they are there. Running on both is a cross-check: the numbers should agree.

Missing readings come back from the authors' `evaluate()` as tiny residues rather than exact zeros; the script treats |truth| < 1e-3 as missing and prints how many it found.

In [ ]:
!python {SCRIPTS}/coverage_conformal.py --selftest --out /content/selftest | tail -3

In [ ]:
# 4a. Our own predictions from the checkpoints (kept on local disk; results go to Drive)
PRED_DIR = '/content/predictions'
!python {SCRIPTS}/make_predictions.py --dataset_dir {PEMSD4_DIR} {args} --out {PRED_DIR}
own = sorted(glob.glob(f'{PRED_DIR}/*.npz'))
print(own)
npz_args = ' '.join(f'--npz {os.path.splitext(os.path.basename(f))[0]}={f}' for f in own)
!python {SCRIPTS}/coverage_conformal.py {npz_args} --out {OUT}/coverage
for f in sorted(glob.glob(f'{OUT}/coverage/*.png')):
    print(f); display(Image(f))

In [ ]:
# 4b. Cross-check on the shared prediction files, if present.
# New folders on the shared Drive can't be written to right after creation, so write locally, then copy into an existing folder.
import shutil
shared = sorted(glob.glob(f'{ROOT}/outputs/pemsd4/*.npz'))
print(shared)
if shared:
    npz_args = ' '.join(f'--npz {os.path.splitext(os.path.basename(f))[0]}={f}' for f in shared)
    !python {SCRIPTS}/coverage_conformal.py {npz_args} --out /content/coverage_shared
    for f in glob.glob('/content/coverage_shared/*'):
        shutil.copy(f, f'{OUT}/coverage/shared_' + os.path.basename(f))